# RanPAC Critique C · Độ nhạy theo Chất lượng & Mục tiêu huấn luyện của Pre-trained Backbone

Notebook này thuộc phần **Critical Analysis & Empirical Demonstration** trên **Split CIFAR-100** (`seed = 1993`, 10 task × 10 lớp), kiểm chứng trực tiếp giả định nền tảng thứ hai của bài báo:

> **Giả định về không gian biểu diễn tiền huấn luyện:** Ở Phase 2, toàn bộ backbone bị đóng băng và lớp Random Projection $H = \operatorname{ReLU}(F W)$ chỉ thực hiện ánh xạ hạt nhân ngẫu nhiên cố định (không học đặc trưng mới). Do đó, trần hiệu năng của RanPAC phụ thuộc trực tiếp vào độ tách biệt tuyến tính/góc sẵn có của backbone tiền huấn luyện.

### Thiết kế đối chứng có kiểm soát 1 nhân tố (One-Factor Controlled Comparison)

| Mã Backbone | Mô hình `timm` | Chiều $d$ | Chế độ Pre-train | Nhân tố kiểm chứng so với Chuẩn Paper |
| --- | --- | ---: | --- | --- |
| `vit_b16_in21k` | `vit_base_patch16_224.augreg_in21k` | 768 | Supervised ImageNet-21K (14M ảnh, 21k lớp) | **Chuẩn tham chiếu bài báo** (tái sử dụng cache sẵn có). |
| `vit_b16_in1k` | `vit_base_patch16_224.augreg_in1k` | 768 | Supervised ImageNet-1K (1.3M ảnh, 1k lớp) | Giữ nguyên kiến trúc ViT-B/16, **thu hẹp dữ liệu pre-train 21K $\to$ 1K**. |
| `vit_b16_dino` | `vit_base_patch16_224.dino` | 768 | Self-Supervised DINO trên IN-1K (không nhãn) | Giữ nguyên kiến trúc ViT-B/16, **bỏ giám sát nhãn lớp khi pre-train**. |
| `vit_t16_in21k` | `vit_tiny_patch16_224.augreg_in21k` | 192 | Supervised ImageNet-21K (5.5M tham số) | Giữ nguyên nguồn dữ liệu IN-21K, **giảm dung lượng mô hình 86M $\to$ 5.5M**. |

Trên **mỗi backbone**, pipeline chạy đủ 3 bộ phân loại Phase 2 (`NCM cosine`, `Ridge không RP`, `RanPAC RP M=10000`) với **Task-CV $\lambda$** (tự động thích nghi $\lambda_t$ theo thang chuẩn đặc trưng riêng của từng backbone để loại trừ sai lệch do cố định $\lambda$), đồng thời đo **Tỷ số Fisher $\operatorname{Tr}(S_B)/\operatorname{Tr}(S_W)$** và **Biên góc (Angular Margin)** của không gian đặc trưng gốc.


## 1. Cài đặt thư viện và kết nối Google Drive

Đặt `ranpac_core.py`, `ranpac_data.py`, `ranpac_colab.py`, `ranpac_full.py`, `ranpac_critique.py` và `test_ranpac_core.py` trong `MyDrive/RanPAC_implement`. Chọn **GPU runtime** để trích xuất nhanh đặc trưng của 3 backbone đối chứng (~2 phút/backbone) và giải Task-CV `M = 10000` trên GPU trong vài giây.


In [ ]:
%pip install -q timm==1.0.22 wandb==0.31.0


In [ ]:
import gc
import sys
import unittest
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')
SOURCE_DIR = Path('/content/drive/MyDrive/RanPAC_implement')
WORK_DIR = Path('/content/drive/MyDrive/ranpac-runs')
DATA_ROOT = '/content/ranpac-data'

required = (
    'ranpac_core.py', 'ranpac_data.py', 'ranpac_colab.py',
    'ranpac_full.py', 'ranpac_critique.py', 'test_ranpac_core.py',
)
CODE_DIR = SOURCE_DIR / 'src' if (SOURCE_DIR / 'src' / 'ranpac_core.py').is_file() else SOURCE_DIR
missing = [name for name in required if not (CODE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Thiếu {", ".join(missing)} trong {CODE_DIR}')
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from ranpac_core import SEED
from ranpac_critique import BACKBONE_SPECS, run_backbone_suite
from ranpac_data import atomic_output


In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromName('test_ranpac_core')
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError('Kiểm tra unit test trên CPU chưa đạt.')


## 2. Cấu hình danh sách Backbone và Tham số Phase 2

- Mặc định `FIXED_LAMBDA = None` bật **80/20 Task-CV $\lambda$** giống hệt Table 1, đảm bảo sự công bằng tuyệt đối khi chiều đặc trưng đổi từ `768` sang `192` hoặc khi chuẩn LayerNorm của `DINO` khác với `AugReg`.
- `STATS_DEVICE = 'cuda'` tận dụng GPU cho phép phân rã trị riêng dual ridge sau khi đã giải phóng backbone khỏi VRAM (mỗi cấu hình `M = 10000` chỉ mất ~8–12 giây).


In [ ]:
WANDB_ENTITY = 'pnp123'
WANDB_PROJECT = 'RanPAC-reproduction'
WANDB_MODE = 'online'          # 'online', 'offline', hoặc 'disabled'
RUN_LABEL = 'bb-01'

SELECTED_BACKBONES = (
    'vit_b16_in21k',
    'vit_b16_in1k',
    'vit_b16_dino',
    'vit_t16_in21k',
)
M = 10_000
FIXED_LAMBDA = None            # None = chọn lambda bằng 80/20 CV theo từng task
BATCH_SIZE = 64
NUM_WORKERS = 2
STATS_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU runtime trên Colab để trích xuất đặc trưng của các backbone đối chứng.')

display(pd.DataFrame([
    {'key': k, **{ col: BACKBONE_SPECS[k][col] for col in ('title', 'model_id', 'feature_dim', 'pretrain_regime') }}
    for k in SELECTED_BACKBONES
]))


## 3. Trích xuất đặc trưng (có Cache) và Đánh giá bộ 3 Classifier trên từng Backbone


In [ ]:
backbone_rows = []
for bb_key in SELECTED_BACKBONES:
    print(f'\n================ {BACKBONE_SPECS[bb_key]["title"]} ================')
    res = run_backbone_suite(
        bb_key, WORK_DIR,
        data_root=DATA_ROOT,
        projection_dim=M,
        fixed_lambda=FIXED_LAMBDA,
        batch_size=BATCH_SIZE,
        workers=NUM_WORKERS,
        stats_device=STATS_DEVICE,
        run_label=RUN_LABEL,
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        wandb_mode=WANDB_MODE,
    )
    backbone_rows.append({k: v for k, v in res.items() if k != 'runs'})
    gc.collect()
    torch.cuda.empty_cache()


## 4. Bảng đối chiếu tổng hợp và Tương quan Hình học Không gian Đặc trưng

- `fisher_trace_ratio` ($\operatorname{Tr}(S_B)/\operatorname{Tr}(S_W)$) và `angular_margin` đo mức độ gom cụm nội lớp so với tách biệt liên lớp ngay trên không gian đặc trưng đóng băng chưa qua RP.
- `drop_vs_in21k_pp`: mức sụt giảm độ chính xác cuối cùng `A[10]` của RanPAC khi thay đổi nguồn dữ liệu pre-train, hàm mục tiêu tự giám sát hoặc dung lượng mô hình.


In [ ]:
df_bb = pd.DataFrame(backbone_rows)
ref_acc = float(df_bb.loc[df_bb['backbone_key'] == 'vit_b16_in21k', 'ranpac_rp_a10_pct'].iloc[0])
df_bb['drop_vs_in21k_pp'] = df_bb['ranpac_rp_a10_pct'] - ref_acc

summary_dir = WORK_DIR / 'sweeps' / f'backbone-sensitivity-s{SEED}-{RUN_LABEL}'
with atomic_output(summary_dir / 'backbone_sensitivity_summary.csv') as tmp:
    df_bb.to_csv(tmp, index=False)

cols_show = [
    'backbone_key', 'feature_dim', 'fisher_trace_ratio', 'angular_margin',
    'ncm_a10_pct', 'ridge_no_rp_a10_pct', 'ranpac_rp_a10_pct',
    'delta_rp_vs_ridge_pp', 'rel_error_reduction_pct', 'drop_vs_in21k_pp',
]
display(df_bb[cols_show].round(3))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6), layout='constrained')
x = np.arange(len(df_bb))
w = 0.26

# Panel 1: Grouped bars of NCM vs Ridge vs RanPAC RP across backbones
ax1 = axes[0]
b_ncm = ax1.bar(x - w, df_bb['ncm_a10_pct'], w, label='NCM cosine', color='#94a3b8')
b_rdg = ax1.bar(x, df_bb['ridge_no_rp_a10_pct'], w, label='Ridge (No RP)', color='#f59e0b')
b_rp = ax1.bar(x + w, df_bb['ranpac_rp_a10_pct'], w, label=f'RanPAC (RP M={M})', color='#2563eb')
ax1.set_xticks(x)
ax1.set_xticklabels(df_bb['backbone_key'], rotation=12)
ax1.set_ylabel('Final Average Accuracy A[10] (%)')
ax1.set_title('Sự phụ thuộc của RanPAC vào Pre-trained Backbone')
ax1.set_ylim(max(40.0, df_bb['ncm_a10_pct'].min() - 6.0), 95.0)
ax1.grid(axis='y', alpha=0.3)
ax1.legend(fontsize=9)
for bar in (*b_ncm, *b_rdg, *b_rp):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)

# Panel 2: Pre-trained Fisher Ratio vs RanPAC A[10]
ax2 = axes[1]
ax2.scatter(df_bb['fisher_trace_ratio'], df_bb['ranpac_rp_a10_pct'],
            s=95, color='#dc2626', zorder=3)
for _, row in df_bb.iterrows():
    ax2.annotate(
        f"{row['backbone_key']}\n(ΔRP={row['delta_rp_vs_ridge_pp']:+.1f}pp)",
        (row['fisher_trace_ratio'], row['ranpac_rp_a10_pct']),
        textcoords='offset points', xytext=(6, -4), fontsize=8.5,
    )
ax2.set_xlabel('Pre-trained Fisher Trace Ratio Tr(S_B) / Tr(S_W)')
ax2.set_ylabel('RanPAC Phase 2 A[10] (%)')
ax2.set_title('Trần hiệu năng bị quy định bởi hình học đặc trưng tiền huấn luyện')
ax2.grid(True, alpha=0.3)

fig_path = summary_dir / 'backbone_sensitivity_analysis.png'
with atomic_output(fig_path) as tmp:
    fig.savefig(tmp, dpi=180, format='png')
plt.show()
print(f'Đã lưu bảng và biểu đồ tại: {summary_dir}')
